In [91]:
import os, sys
from typing import Any
from pandas import Series, DataFrame
from pandas import Timestamp, Timedelta
from pandas import read_pickle, concat
from pandas import Index, MultiIndex

tests = dict[str, Any]()
for fn in os.listdir("../tests/"):
    if fn.endswith(".pickle"):
        key = fn[: 14]
        if key not in tests:
            tests[key] = list[str]()
        list.append(tests[key], fn)

print("Available tests:")
for key, files in tests.items():
    print(f" >> {key}: ({len(files)} files) ... {files}")
file_keys = ["ticks_p", "candles_p", "ticks_b", "candles_b"]

Available tests:
 >> 20260511224819: (4 files) ... ['20260511224819_candles_b.pickle', '20260511224819_candles_p.pickle', '20260511224819_ticks_b.pickle', '20260511224819_ticks_p.pickle']


In [73]:
key = list(tests.keys())[-1]
dftb = read_pickle(f"../tests/{key}_ticks_b.pickle")
dftp = read_pickle(f"../tests/{key}_ticks_p.pickle")
dfcb = read_pickle(f"../tests/{key}_candles_b.pickle")
dfcp = read_pickle(f"../tests/{key}_candles_p.pickle")
min_p = int(dftb["pb"].min().min())
side = "a"
dftb = dftb.reset_index(["venue", "symbol"], drop = True)[["pa", "pb"]].astype(int) - min_p
dftp = dftp.reset_index(["venue", "symbol"], drop = True)[["pa", "pb"]].astype(int) - min_p
dfcb[Index([*"ohlc"]) + "a"] = dfcb[Index([*"ohlc"]) + "a"].astype(int) - min_p
dfcb[Index([*"ohlc"]) + "b"] = dfcb[Index([*"ohlc"]) + "b"].astype(int) - min_p
dfcp[Index([*"ohlc"]) + "a"] = dfcp[Index([*"ohlc"]) + "a"].astype(int) - min_p
dfcp[Index([*"ohlc"]) + "b"] = dfcp[Index([*"ohlc"]) + "b"].astype(int) - min_p
p_columns = Index([*"ohlc"]) + side
dfcb = dfcb.reset_index(["venue", "symbol"], drop = True)[[*p_columns, "volume"]]
dfcp = dfcp.reset_index(["venue", "symbol"], drop = True)[[*p_columns, "volume"]]
#DataFrame.merge(dftb, dftp, left_index = True, right_index = True, how = "outer")
dftb.index = dftb.index.floor("1s")
dftb = dftb.groupby("time")["p" + side].apply(list)
df = DataFrame.merge(dfcb.loc["S1"], dftb,
        left_index = True, right_index = True,
        how = "outer").iloc[1 : -1]

{
    "o": (df["o" + side] == df["p" + side].str[0]).all(),
    "h": (df["h" + side] == df["p" + side].map(max)).all(),
    "l": (df["l" + side] == df["p" + side].map(min)).all(),
    "c": (df["c" + side] == df["p" + side].str[-1]).all(),
    "v": (df["volume"] == df["p" + side].map(len)).all()
}

{'o': np.True_, 'h': np.True_, 'l': np.True_, 'c': np.True_, 'v': np.True_}

In [77]:
key = list(tests.keys())[-1]
dftb = read_pickle(f"../tests/{key}_ticks_b.pickle")
dftp = read_pickle(f"../tests/{key}_ticks_p.pickle")
dfcb = read_pickle(f"../tests/{key}_candles_b.pickle")
dfcp = read_pickle(f"../tests/{key}_candles_p.pickle")
dfcb = dfcb.reset_index(["venue", "symbol"], drop = True)
dfcp = dfcp.reset_index(["venue", "symbol"], drop = True)

In [78]:
dfcb

volume       oa        ha       la       ca  \
tf time                                                                     
H1 2026-05-11 01:00:00+00:00    7171  99877.0   99909.0  99727.0  99744.0   
   2026-05-11 02:00:00+00:00    7181  99756.0   99850.0  99719.0  99830.0   
   2026-05-11 03:00:00+00:00    7233  99845.0   99862.0  99721.0  99817.0   
   2026-05-11 04:00:00+00:00    7239  99823.0   99982.0  99765.0  99950.0   
   2026-05-11 05:00:00+00:00    7223  99955.0  100047.0  99881.0  99950.0   
...                              ...      ...       ...      ...      ...   
S6 2026-05-11 13:53:00+00:00      14  99776.0   99776.0  99758.0  99764.0   
   2026-05-11 13:53:06+00:00      10  99756.0   99769.0  99756.0  99769.0   
   2026-05-11 13:53:12+00:00      13  99764.0   99772.0  99752.0  99752.0   
   2026-05-11 13:53:18+00:00      11  99759.0   99765.0  99754.0  99761.0   
   2026-05-11 13:53:24+00:00      11  99755.0   99763.0  99742.0  99760.0   

                                   ob        hb       lb       cb  
tf time                                                            
H1 2026-05-11 01:00:00+00:00  99871.0   99892.0  99724.0  99735.0  
   2026-05-11 02:00:00+00:00  99736.0   99832.0  99715.0  99830.0  
   2026-05-11 03:00:00+00:00  99829.0   99845.0  99720.0  99809.0  
   2026-05-11 04:00:00+00:00  99808.0   99963.0  99763.0  99950.0  
   2026-05-11 05:00:00+00:00  99951.0  100031.0  99879.0  99949.0  
...                               ...       ...      ...      ...  
S6 2026-05-11 13:53:00+00:00  99761.0   99762.0  99754.0  99754.0  
   2026-05-11 13:53:06+00:00  99755.0   99758.0  99754.0  99758.0  
   2026-05-11 13:53:12+00:00  99757.0   99757.0  99750.0  99751.0  
   2026-05-11 13:53:18+00:00  99750.0   99750.0  99745.0  99746.0  
   2026-05-11 13:53:24+00:00  99745.0   99745.0  99742.0  99743.0  

[141513 rows x 9 columns]

In [102]:
df = DataFrame.merge(dfcb.loc["S1"], dfcp.loc["S1"],
    left_index = True, right_index = True,
    how = "outer", suffixes = ["_b", "_p"])

columns = df.columns.copy()
df.columns = [tuple(str.split(c, "_")) for c in columns]
df.columns = MultiIndex.from_tuples(df.columns).swaplevel(0, 1)
df.columns = df.columns.rename(["source", "field"])
df = df.dropna().astype(int).stack("field").sort_index()
(df["b"] / df["p"] - 1).groupby("field").mean()

C:\Users\GSL\AppData\Local\Temp\ipykernel_29856\1373231948.py:9: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df = df.dropna().astype(int).stack("field").sort_index()


field
ca        0.0
cb        0.0
ha        0.0
hb        0.0
la        0.0
lb        0.0
oa        0.0
ob        0.0
volume    0.0
dtype: float64